# Phase 3B — ViT (SwinUNETR) Embedding Evaluation Battery (18 Tests)
## Quantitative Evaluation: ViT vs CNN Baseline Comparison

**Model**: SwinUNETR (BraTS 2021 Fold 1 pretrained → BraTS 2024 fine-tuned)
**Best Dice**: 0.8379 (epoch 19) — WT=0.870 TC=0.815 ET=0.829

**Embedding**: 2121-D multi-scale
- Component 1: 8×192 = 1536-D octant spatial pool (layers2[0] → 16×16×16, ROI-cropped)
- Component 2: 3×192 = 576-D mask-weighted pool (WT + TC + ET within ROI)
- Component 3: 9-D volumetric (log_vol × 3 + presence_flag × 3 + ratio × 3)

**Tests:** Same 18 tests / 26 metrics as Phase 2 CNN battery:
- M1-M6: Morphology (tumour size, necrosis, core fraction, patient purity)
- H1-H5: Heterogeneity (RankMe, Diversity, Uniformity, Responder F1, Norm CV)
- T1-T8: Temporal (Spearman rho, ordering, RANO AUC, coherence, Kendall tau)

**Purpose:** Prove ViT improves over CNN — especially M6 (patient purity), T4 (RANO AUC), T7 (Cohen's d), T8 (Kendall tau).

**CNN Baseline (Phase 2):** 16/26 pass | M6=7.3% | T4=0.526 | T8=0.251

In [ ]:
import numpy as np, json, random, warnings
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import r2_score, f1_score, roc_auc_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr, spearmanr
from scipy.stats import kendalltau as kt
import pandas as pd
warnings.filterwarnings("ignore")

OUTPUT_ROOT = Path("/kaggle/working/phase3_evaluation")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ── Load nnUNet embeddings ──
models = {}
# Priority: v4 (triplet-trained) → v3 (layers3) → base
EMB_NAMES = ["vit_swinunetr_embeddings_v4", "vit_swinunetr_embeddings_v3", "vit_swinunetr_embeddings"]
SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]
for emb_name in EMB_NAMES:
    for root in SEARCH_ROOTS:
        matches = list(root.rglob(f"{emb_name}.npz"))
        if matches:
            data = np.load(matches[0], allow_pickle=True)
            embs_arr  = data["embeddings"]          # (N, 1280)
            pids_arr  = data["patient_ids"]         # (N,)
            tps_arr   = data["timepoints"]          # (N,)
            # Build dict keyed by "patientid__timepoint"
            emb_dict = {}
            for i in range(len(embs_arr)):
                key = "{}__{}".format(str(pids_arr[i]), str(tps_arr[i]))
                emb_dict[key] = embs_arr[i]
            models["swinunetr"] = emb_dict
            print("swinunetr: {} scans | dim={} | file={}".format(
                len(emb_dict), embs_arr.shape[1], matches[0]))
            break
    if "swinunetr" in models:
        break

if not models:
    raise FileNotFoundError(
        "No swinunetr embeddings found. Attach the dataset containing "
        "vit_swinunetr_embeddings.npz as input.")

# ── Load tumour volume metadata (if available) ──
tumor_df = None
for f in (list(Path("/kaggle/input").rglob("tumor_volumes.csv")) +
          list(Path("/kaggle/working").rglob("tumor_volumes.csv"))):
    tumor_df = pd.read_csv(f); break

# ── Load clinical metadata (if available) ──
meta_df = None
for f in (list(Path("/kaggle/input").rglob("*.xlsx")) +
          list(Path("/kaggle/input").rglob("*.csv"))):
    try:
        df = pd.read_excel(f) if str(f).endswith(".xlsx") else pd.read_csv(f)
        if any("BraTS" in str(c) for c in df.columns):
            meta_df = df; break
    except Exception:
        continue

print("Models: {} | Tumor vols: {} | Clinical meta: {}".format(
    list(models.keys()), tumor_df is not None, meta_df is not None))

# ── Quick sanity printout ──
for mn, embs in models.items():
    keys = list(embs.keys())
    norms = np.linalg.norm(np.stack([embs[k] for k in keys]), axis=1)
    pids  = [k.split("__")[0] for k in keys]
    tps   = [k.split("__")[1] for k in keys]
    print("{}: {} scans | {} patients | timepoints {} -> {}".format(
        mn, len(keys), len(set(pids)), min(tps), max(tps)))
    print("  Norm: [{:.3f}, {:.3f}]  mean={:.3f}".format(
        norms.min(), norms.max(), norms.mean()))

# ── Debug tumor_df columns ──
if tumor_df is not None:
    print("tumor_df shape:", tumor_df.shape)
    print("tumor_df columns:", list(tumor_df.columns))
    print("tumor_df sample:")
    print(tumor_df.head(3).to_string())

# ── Flexible matching helper ──
# Handles mismatches: int vs str timepoints, pid prefix differences
def match_row(tumor_df, pid, tp):
    """Find a row in tumor_df matching this patient + timepoint."""
    if tumor_df is None:
        return None
    # Try exact match first
    m = tumor_df[(tumor_df["patient_id"].astype(str) == str(pid)) &
                 (tumor_df["timepoint"].astype(str) == str(tp))]
    if len(m) > 0:
        return m.iloc[0]
    # Try numeric timepoint (100->0, 101->1 ...)
    tp_int = int(tp) - 100 if str(tp).isdigit() and int(tp) >= 100 else int(tp)
    m = tumor_df[(tumor_df["patient_id"].astype(str) == str(pid)) &
                 (tumor_df["timepoint"].astype(str) == str(tp_int))]
    if len(m) > 0:
        return m.iloc[0]
    # Try partial pid match (in case prefix differs)
    pid_short = str(pid).split("-")[-1] if "-" in str(pid) else str(pid)
    for col in ["patient_id", "subject_id", "BraTS Subject ID", "ID"]:
        if col not in tumor_df.columns:
            continue
        m = tumor_df[tumor_df[col].astype(str).str.contains(pid_short, na=False)]
        if len(m) > 0:
            return m.iloc[0]
    return None

# Quick match test
if tumor_df is not None:
    keys = list(list(models.values())[0].keys())
    pid0, tp0 = keys[0].split("__")
    r = match_row(tumor_df, pid0, tp0)
    print("Match test for key {}: {}".format(keys[0], "FOUND" if r is not None else "NOT FOUND"))
    if r is not None:
        print("  Row:", dict(r))

# ── 1929-D structure info ──
# dims  0:768   = global context (layers4[0])
# dims 768:1152  = WT region-pooled (layers3[0])
# dims 1152:1536 = TC region-pooled
# dims 1536:1920 = ET region-pooled
# dims 1920:1929 = volumetric [log_wt,log_tc,log_et,has_wt,has_tc,has_et,tc_wt,et_wt,et_tc]
VOL_DIMS = slice(1920, 1929)  # can inspect these directly
for mn, embs in models.items():
    keys = list(embs.keys())
    arr  = np.stack([embs[k] for k in keys])
    if arr.shape[1] >= 1929:
        vol = arr[:, 1920:1929]
        print(f"{mn}: vol features sample (scan 0): "
              f"log_wt={vol[0,0]:.2f} log_tc={vol[0,1]:.2f} log_et={vol[0,2]:.2f} "
              f"has_wt={vol[0,3]:.0f} has_tc={vol[0,4]:.0f} has_et={vol[0,5]:.0f} "
              f"tc_wt={vol[0,6]:.2f} et_wt={vol[0,7]:.2f} et_tc={vol[0,8]:.2f}")

# ═══════════════════════════════════════════════════════════════
# COMPONENT-WISE L2 NORMALISATION
# ═══════════════════════════════════════════════════════════════
# Problem: The 768-D global vector from layers4 encodes brain anatomy
#   (nearly identical across all patients), dominating the embedding.
#   Result: cosine sim ≈ 0.98 for ALL pairs → collapsed diversity.
#
# Fix: L2-normalise each component INDEPENDENTLY so that:
#   - Global (768-D): brain-level features on unit sphere
#   - Region (3×384-D): tumor-specific features on unit sphere
#   - Volumetric (9-D): explicit morphology on unit sphere
#   This gives each component EQUAL geometric weight.
#
# Applied BEFORE all 26 metrics. This is the standard approach
# for multi-scale concatenated embeddings (cf. Wang & Isola 2020).
# ═══════════════════════════════════════════════════════════════

for mn in list(models.keys()):
    emb_dict = models[mn]
    keys = list(emb_dict.keys())
    arr  = np.stack([emb_dict[k] for k in keys])
    D = arr.shape[1]
    print(f'\n{mn}: Component-wise L2 normalisation (D={D})')

    if D >= 2121:  # v2 ROI-crop + octant format
        comp_octant = arr[:, 0:D-576-9]       # octant: 8×C (1536 if C=192)
        comp_region = arr[:, D-576-9:D-9]      # region: 3×C (576 if C=192)
        comp_vol    = arr[:, D-9:]              # volumetric: 9-D

        # Print pre-normalisation diagnostics
        o_norms = np.linalg.norm(comp_octant, axis=1)
        r_norms = np.linalg.norm(comp_region, axis=1)
        v_norms = np.linalg.norm(comp_vol, axis=1)
        print(f'  BEFORE norm:')
        print(f'    Octant ({comp_octant.shape[1]}-D): mean={o_norms.mean():.1f}  CV={o_norms.std()/(o_norms.mean()+1e-8):.3f}')
        print(f'    Region ({comp_region.shape[1]}-D): mean={r_norms.mean():.1f}  CV={r_norms.std()/(r_norms.mean()+1e-8):.3f}')
        print(f'    Vol      (9-D):  mean={v_norms.mean():.1f}  CV={v_norms.std()/(v_norms.mean()+1e-8):.3f}')

        # L2-normalise each component independently
        comp_octant_n = comp_octant / (np.linalg.norm(comp_octant, axis=1, keepdims=True) + 1e-8)
        comp_region_n = comp_region / (np.linalg.norm(comp_region, axis=1, keepdims=True) + 1e-8)
        comp_vol_n    = comp_vol    / (np.linalg.norm(comp_vol,    axis=1, keepdims=True) + 1e-8)

        # ── v2: All components are tumor-focused (no brain anatomy noise) ──
        # Octant: spatial heterogeneity within tumor ROI
        # Region: WT/TC/ET mask-weighted features
        # Vol: explicit morphology (log-vol, presence, ratios)
        arr_balanced = np.concatenate([comp_octant_n, comp_region_n, comp_vol_n * 2], axis=1)
        print(f'  Balanced embedding: D={arr_balanced.shape[1]}')

        # Post-normalisation diagnostics
        full_norms = np.linalg.norm(arr_balanced, axis=1)
        print(f'  AFTER norm:')
        print(f'    Full ({arr_balanced.shape[1]}-D): mean={full_norms.mean():.3f}  CV={full_norms.std()/full_norms.mean():.4f}')

        # Check cosine diversity after fix
        idx_pairs = np.random.choice(len(keys), (500, 2))
        arr_n = arr_balanced / (np.linalg.norm(arr_balanced, axis=1, keepdims=True) + 1e-8)
        sims = (arr_n[idx_pairs[:,0]] * arr_n[idx_pairs[:,1]]).sum(1)
        print(f'    Cosine sim (500 pairs): mean={sims.mean():.3f}  std={sims.std():.3f}')
        print(f'    Diversity: {1-sims.mean():.3f}  (was 0.018 pre-fix)')

        # Replace embeddings with balanced version
        models[mn] = {k: arr_balanced[i] for i, k in enumerate(keys)}
        print(f'  ✅ Embeddings replaced with component-normalised version')
    else:
        # Standard single-scale embedding — just L2-normalise globally
        arr_n = arr / (np.linalg.norm(arr, axis=1, keepdims=True) + 1e-8)
        models[mn] = {k: arr_n[i] for i, k in enumerate(keys)}
        print(f'  Standard L2 normalisation applied (D={D})')


# Initialise results dict (used by all test cells)
results = {}


In [ ]:
# M1-M6: MORPHOLOGY TESTS (REVISED — nonlinear probes + L2 norm)
print("=" * 60)
print("  MORPHOLOGY TESTS M1-M6")
print("=" * 60)

for mn, embs in models.items():
    keys = list(embs.keys())
    X    = np.stack([embs[k] for k in keys])
    # ── L2 normalise (recommendation: fix H4 norm variance) ──
    X_l2 = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
    Xs   = StandardScaler().fit_transform(X_l2)  # scale L2-normed
    if mn not in results: results[mn] = {}

    mX, mvols = [], {"wt": [], "tc": [], "et": []}
    for k in keys:
        pid, tp = k.split("__")
        row = match_row(tumor_df, pid, tp)
        if row is not None:
            mX.append(Xs[keys.index(k)])
            for r_name in ["wt", "tc", "et"]:
                for col in [f"{r_name}_vol", f"{r_name.upper()}_vol",
                             f"{r_name}_volume", f"vol_{r_name}"]:
                    if col in row.index:
                        mvols[r_name].append(float(row[col])); break
                else:
                    mvols[r_name].append(0.0)

    print("{}: {}/{} matched".format(mn, len(mX), len(keys)))
    if len(mX) < 10:
        print("  Too few matched — skipping regression tests")
        # M6 still runs (label-free)
        pids_arr = np.array([k.split("__")[0] for k in keys])
        nbrs = NearestNeighbors(n_neighbors=11).fit(Xs)
        _, idx = nbrs.kneighbors(Xs)
        results[mn]["M6_patient_purity_pct"] = float(
            100 * np.mean([np.mean(pids_arr[idx[i,1:]] == pids_arr[i])
                           for i in range(len(keys))]))
        continue

    mX = np.stack(mX)
    y_wt = np.array(mvols["wt"])
    y_tc = np.array(mvols["tc"])
    y_et = np.array(mvols["et"])
    ridge = Ridge(alpha=1.0)
    rf    = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

    # ── M1: Volume R² — Ridge (linear) + RF (nonlinear) + Spearman ──
    p_ridge = cross_val_predict(ridge, mX, y_wt, cv=5)
    p_rf    = cross_val_predict(rf,    mX, y_wt, cv=5)
    results[mn]["M1_volume_R2_ridge"]  = float(r2_score(y_wt, p_ridge))  # expected negative
    results[mn]["M1_volume_R2_rf"]     = float(r2_score(y_wt, p_rf))     # nonlinear probe
    results[mn]["M1_spearman_rho"]     = float(spearmanr(p_ridge, y_wt)[0])

    # ── M2: Log-Volume R² — Ridge + RF ──
    yl = np.log1p(y_wt)
    results[mn]["M2_logvol_R2_ridge"] = float(r2_score(yl, cross_val_predict(ridge, mX, yl, cv=5)))
    results[mn]["M2_logvol_R2_rf"]    = float(r2_score(yl, cross_val_predict(rf,    mX, yl, cv=5)))

    # ── M3: Enhancement Fraction — Ridge + RF ──
    y_ef = y_et / (y_wt + 0.01)
    results[mn]["M3_enhancement_ridge"] = float(r2_score(y_ef, cross_val_predict(ridge, mX, y_ef, cv=5)))
    results[mn]["M3_enhancement_rf"]    = float(r2_score(y_ef, cross_val_predict(rf,    mX, y_ef, cv=5)))

    # ── M4: Necrosis F1 (unchanged) ──
    ncr   = y_tc - y_et
    y_ncr = ((ncr / (y_tc + 1e-6)) > 0.10).astype(int)
    if len(set(y_ncr)) >= 2:
        p_ncr = cross_val_predict(LogisticRegression(max_iter=1000), mX, y_ncr, cv=5)
        results[mn]["M4_necrosis_F1"] = float(f1_score(y_ncr, p_ncr, average="weighted"))
    else:
        results[mn]["M4_necrosis_F1"] = 0.0

    # ── M5: Core Fraction — Ridge + RF ──
    y_cf = y_tc / (y_wt + 0.01)
    results[mn]["M5_corefrac_ridge"] = float(r2_score(y_cf, cross_val_predict(ridge, mX, y_cf, cv=5)))
    results[mn]["M5_corefrac_rf"]    = float(r2_score(y_cf, cross_val_predict(rf,    mX, y_cf, cv=5)))

    # ── M6: Patient Identity Purity (label-free, on L2-normed space) ──
    pids_arr = np.array([k.split("__")[0] for k in keys])
    nbrs = NearestNeighbors(n_neighbors=11).fit(Xs)
    _, idx = nbrs.kneighbors(Xs)
    results[mn]["M6_patient_purity_pct"] = float(
        100 * np.mean([np.mean(pids_arr[idx[i,1:]] == pids_arr[i])
                       for i in range(len(keys))]))

    for k, v in sorted(results[mn].items()):
        if k.startswith("M"):
            print("  {} {}: {:.3f}".format(mn, k, v))

# ── CNN vs ViT comparison for morphology ──
CNN_BASELINE_M = {
    'M1_volume_R2_rf': 0.576, 'M1_spearman_rho': 0.611,
    'M2_logvol_R2_rf': 0.656, 'M3_enhancement_rf': 0.357,
    'M4_necrosis_F1': 0.748,  'M5_corefrac_rf': 0.310,
    'M6_patient_purity_pct': 7.278,
}
print('\nCNN vs ViT — Morphology:')
for mn, res in results.items():
    for k, cnn_v in CNN_BASELINE_M.items():
        vit_v = res.get(k, float('nan'))
        arrow = '✅ ↑' if vit_v > cnn_v else ('❌ ↓' if vit_v < cnn_v else '=')
        print(f'  {k:<30} CNN={cnn_v:.3f}  ViT={vit_v:.3f}  {arrow}')


In [ ]:
# H1-H5: HETEROGENEITY TESTS (REVISED)
# H1: RankMe + effective rank (replaces PCA range)
# H2: 2000 pairs + Uniformity (Wang & Isola 2020)
# H3: Treatment responder F1 (replaces IDH)
# H5: RankMe standalone (new)
print('\n' + '='*60)
print('  HETEROGENEITY TESTS H1-H5')
print('='*60)

for mn, embs in models.items():
    keys = list(embs.keys())
    X    = np.stack([embs[k] for k in keys])
    # L2-normalise before all heterogeneity metrics
    X_l2 = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
    Xs   = StandardScaler().fit_transform(X_l2)

    # H1 — RankMe + Effective Rank (replaces PCA cumulative variance)
    U, S, Vh = np.linalg.svd(Xs, full_matrices=False)
    p_sv = S / S.sum()
    rankme = float(np.exp(-np.sum(p_sv * np.log(p_sv + 1e-12))))
    eff_rank_95 = int(np.searchsorted(np.cumsum(p_sv), 0.95)) + 1
    results[mn]['H1_rankme']       = rankme
    results[mn]['H1_eff_rank_95']  = float(eff_rank_95)

    # H2 — Diversity (2000 pairs) + Uniformity (Wang & Isola 2020)
    emb_n = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
    idx2  = np.random.choice(len(keys), (2000, 2), replace=True)
    sims  = (emb_n[idx2[:,0]] * emb_n[idx2[:,1]]).sum(1)
    results[mn]['H2_diversity'] = float(1 - np.mean(sims))
    sq_dist = np.sum((emb_n[idx2[:,0]] - emb_n[idx2[:,1]])**2, axis=1)
    results[mn]['H2_uniformity'] = float(np.log(np.mean(np.exp(-2 * sq_dist))))

    # H3 — Treatment Responder F1 (REPLACES IDH)
    # responder if WT volume decreases > 20% from first to last visit
    pe = {}
    for k in keys:
        pid, tp = k.split('__')
        if pid not in pe: pe[pid] = {}
        pe[pid][tp] = embs[k]
    X_resp, y_resp = [], []
    if tumor_df is not None:
        for pid, tps in pe.items():
            if len(tps) < 2: continue
            stps = sorted(tps.keys())
            v0 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint']==int(stps[0]))]
            vT = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint']==int(stps[-1]))]
            if len(v0) > 0 and len(vT) > 0:
                ratio = vT.iloc[0]['wt_vol'] / (v0.iloc[0]['wt_vol'] + 1e-6)
                y_resp.append(1 if ratio < 0.80 else 0)  # responder: -20%
                X_resp.append(Xs[keys.index(f'{pid}__{stps[0]}')])
    if len(X_resp) >= 10 and len(set(y_resp)) >= 2:
        scores = cross_val_score(LogisticRegression(max_iter=1000),
                                 np.stack(X_resp), y_resp, cv=5, scoring='f1_weighted')
        results[mn]['H3_responder_F1'] = float(scores.mean())
    else:
        results[mn]['H3_responder_F1'] = 0.0

    # H4 — Norm CV (raw embeddings)
    norms = np.linalg.norm(X, axis=1)
    results[mn]['H4_norm_cv'] = float(np.std(norms) / (np.mean(norms) + 1e-8))

    # H5 — RankMe standalone (new test)
    results[mn]['H5_rankme_standalone'] = rankme

    for k, v in sorted(results[mn].items()):
        if k.startswith('H'):
            print(f'  {mn} {k}: {v:.3f}')


In [ ]:
# T1-T8: TEMPORAL TESTS (REVISED + T8 NEW)
# T1: Spearman rho per subregion
# T2: Near-duplicate rate (< 1%)
# T3: Delta R2 + Directional AUC
# T4: RANO criterion (ET +40%)
# T5: Coherence dual-bound 0.70-0.93
# T7: Cohen's d on distances (not norms)
# T8: Kendall tau trajectory monotonicity (NEW)
print('\n' + '='*60)
print('  TEMPORAL TESTS T1-T8 (Static Modeling Limitations)')
print('='*60)
from scipy.stats import kendalltau as kt

for mn, embs in models.items():
    keys = list(embs.keys())
    # L2-normalise embeddings before all temporal distance metrics
    raw  = np.stack([embs[k] for k in keys])
    norms_for_l2 = np.linalg.norm(raw, axis=1, keepdims=True) + 1e-8
    embs_l2 = {k: embs[k] / norms_for_l2[i] for i, k in enumerate(keys)}
    pe   = {}
    for k in keys:
        pid, tp = k.split('__')
        if pid not in pe: pe[pid] = {}
        pe[pid][tp] = embs[k]
    longi = {p: t for p, t in pe.items() if len(t) >= 2}
    print(f'  {mn}: {len(longi)} longitudinal patients')
    if len(longi) < 5:
        for t in ['T1','T2','T3','T4','T5','T6','T7','T8']:
            results[mn][t] = 0
        continue

    den, dvol_wt, dvol_et, csim = [], [], [], []
    v_first, v_last = [], []  # for T7

    for pid, tps in longi.items():
        stps = sorted(tps.keys())
        e_first = embs_l2['{}__{}'.format(pid, stps[0])]; e_last = embs_l2['{}__{}'.format(pid, stps[-1])]
        v_first.append(np.linalg.norm(e_last - e_first))  # T7

        for i in range(len(stps) - 1):
            e0 = embs_l2.get('{}__{}'.format(pid, stps[i]),   tps[stps[i]])
            e1 = embs_l2.get('{}__{}'.format(pid, stps[i+1]), tps[stps[i+1]])
            d = np.linalg.norm(e1 - e0)
            den.append(d)
            n0 = np.linalg.norm(e0) + 1e-8
            n1 = np.linalg.norm(e1) + 1e-8
            csim.append(float((e0/n0) @ (e1/n1)))
            if tumor_df is not None:
                v0 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint']==int(stps[i]))]
                v1 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint']==int(stps[i+1]))]
                if len(v0) > 0 and len(v1) > 0:
                    dvol_wt.append(abs(v1.iloc[0]['wt_vol'] - v0.iloc[0]['wt_vol']))
                    et0 = v0.iloc[0]['et_vol']; et1 = v1.iloc[0]['et_vol']
                    dvol_et.append(et1 / (et0 + 1e-6) - 1)  # fractional ET change

    den = np.array(den)

    # T1 — Spearman rho per subregion
    ml = min(len(den), len(dvol_wt))
    if ml >= 5:
        rho_wt, _ = spearmanr(den[:ml], dvol_wt[:ml])
        results[mn]['T1_spearman_wt'] = abs(float(rho_wt))
    else:
        results[mn]['T1_spearman_wt'] = 0

    # T2 — Near-duplicate rate
    near_dup = np.mean(den < 0.001 * den.mean()) if len(den) > 0 else 1.0
    results[mn]['T2_ordering_pass'] = float(near_dup < 0.01)  # 1=pass, 0=fail

    # T3 — Delta R2 + Directional AUC
    ml_et = min(len(den), len(dvol_et))
    if ml_et >= 10:
        yd = np.array(dvol_wt[:ml_et])
        pd3 = cross_val_predict(Ridge(1.0), den[:ml_et].reshape(-1,1), yd, cv=min(5,ml_et//2))
        results[mn]['T3_delta_R2'] = float(r2_score(yd, pd3))  # can be negative
        vol_sign = (np.array(dvol_wt[:ml_et]) > 0).astype(int)
        drift_sign = (den[:ml_et] > np.median(den[:ml_et])).astype(int)
        if len(set(vol_sign)) > 1:
            from sklearn.metrics import roc_auc_score as ras
            results[mn]['T3_directional_auc'] = float(ras(vol_sign, drift_sign))
        else:
            results[mn]['T3_directional_auc'] = 0.5
    else:
        results[mn]['T3_delta_R2'] = 0
        results[mn]['T3_directional_auc'] = 0.5

    # T4 — RANO Response AUC (ET +40%)
    if ml_et >= 10 and tumor_df is not None:
        progressive = (np.array(dvol_et[:ml_et]) > 0.40).astype(int)
        if len(set(progressive)) > 1:
            from sklearn.metrics import roc_auc_score as ras
            results[mn]['T4_rano_auc'] = float(ras(progressive, den[:ml_et]))
        else:
            results[mn]['T4_rano_auc'] = 0.5
    else:
        results[mn]['T4_rano_auc'] = 0.5

    # T5 — Coherence dual-bound
    coherence = float(np.mean(csim)) if csim else 0
    results[mn]['T5_coherence'] = coherence
    results[mn]['T5_pass_dual'] = float(0.70 < coherence < 0.93)

    # T6 — Velocity CV (descriptor)
    results[mn]['T6_velocity_cv'] = float(np.std(den) / (np.mean(den) + 1e-8)) if len(den) else 0

    # T7 — Cohen's d on distances (progressors vs stable)
    all_dists = np.array(v_first)
    if tumor_df is not None and len(all_dists) >= 10:
        pids_longi = list(longi.keys())
        prog_mask = []
        for pid in pids_longi:
            stps = sorted(longi[pid].keys())
            v0 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint']==int(stps[0]))]
            vT = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint']==int(stps[-1]))]
            if len(v0) > 0 and len(vT) > 0:
                prog_mask.append((vT.iloc[0]['et_vol'] / (v0.iloc[0]['et_vol'] + 1e-6) - 1) > 0.40)
            else:
                prog_mask.append(False)
        prog_mask = np.array(prog_mask)
        d_prog = all_dists[prog_mask]; d_stab = all_dists[~prog_mask]
        if len(d_prog) >= 3 and len(d_stab) >= 3:
            pooled = np.sqrt((np.var(d_prog) + np.var(d_stab)) / 2) + 1e-8
            results[mn]['T7_treatment_d'] = float(abs(d_prog.mean() - d_stab.mean()) / pooled)
        else:
            results[mn]['T7_treatment_d'] = 0
    else:
        results[mn]['T7_treatment_d'] = 0

    # T8 — Kendall tau trajectory monotonicity (NEW)
    taus = []
    for pid, tps in longi.items():
        stps = sorted(tps.keys())
        if len(stps) < 3: continue
        dists = [np.linalg.norm(tps[v] - tps[stps[0]]) for v in stps[1:]]
        tau, _ = kt(dists, range(len(dists)))
        taus.append(tau)
    results[mn]['T8_kendall_tau'] = float(np.mean(taus)) if taus else 0

    for k, v in sorted(results[mn].items()):
        if k.startswith('T'):
            flag = ' <- WEAK (static CNN limitation)' if v < 0.30 and k in [
                'T1_spearman_wt','T3_delta_R2','T3_directional_auc','T4_rano_auc','T8_kendall_tau'] else ''
            print(f'  {mn} {k}: {v:.3f}{flag}')

# ── CNN vs ViT comparison for temporal ──
CNN_BASELINE_T = {
    'T1_spearman_wt': 0.387, 'T2_ordering_pass': 1.000,
    'T3_directional_auc': 0.500, 'T4_rano_auc': 0.526,
    'T5_coherence': 0.777, 'T5_pass_dual': 1.000,
    'T7_treatment_d': 0.266, 'T8_kendall_tau': 0.251,
}
print('\nCNN vs ViT — Temporal:')
for mn, res in results.items():
    for k, cnn_v in CNN_BASELINE_T.items():
        vit_v = res.get(k, float('nan'))
        arrow = '✅ ↑' if vit_v > cnn_v else ('❌ ↓' if vit_v < cnn_v else '=')
        print(f'  {k:<30} CNN={cnn_v:.3f}  ViT={vit_v:.3f}  {arrow}')


In [ ]:
# FULL 18-TEST DASHBOARD (updated thresholds + priorities)
print("\n" + "="*60)
print("  FULL 18-TEST DASHBOARD — ViT (SwinUNETR) vs CNN Baseline")
print("="*60)

THRESHOLDS = {
    # Morphology — use RF (nonlinear) as primary probe
    "M1_volume_R2_ridge":     (0.50, "low-pri"),   # expected negative
    "M1_volume_R2_rf":        (0.50, "med-pri"),   # nonlinear probe
    "M1_spearman_rho":        (0.55, "med-pri"),
    "M2_logvol_R2_ridge":     (0.40, "low-pri"),
    "M2_logvol_R2_rf":        (0.40, "med-pri"),
    "M3_enhancement_ridge":   (0.25, "low-pri"),   # TC=ET degeneracy
    "M3_enhancement_rf":      (0.25, "low-pri"),
    "M4_necrosis_F1":         (0.60, "med-pri"),
    "M5_corefrac_ridge":      (0.30, "low-pri"),
    "M5_corefrac_rf":         (0.30, "low-pri"),
    "M6_patient_purity_pct":  (60.0, "HIGH-PRI"),  # biggest Phase3 target
    # Heterogeneity
    "H1_rankme":              (30.0,  "med-pri"),
    "H1_eff_rank_95":         (50.0,  "med-pri"),
    "H2_diversity":            (0.25, "med-pri"),
    "H2_uniformity":          (-3.0,  "med-pri"),
    "H3_responder_F1":         (0.55, "med-pri"),
    "H4_norm_cv":              (0.30, "low-pri"),   # fix: L2-norm applied
    "H5_rankme_standalone":   (30.0,  "med-pri"),
    # Temporal — priority order: T4 > T8 > T1 > T7 > T2 > T5
    "T1_spearman_wt":          (0.30, "HIGH-PRI"),
    "T2_ordering_pass":        (1.0,  "low-pri"),
    "T3_delta_R2":             None,               # removed — inherently hard
    "T3_directional_auc":      (0.55, "low-pri"),
    "T4_rano_auc":             (0.65, "HIGH-PRI"),  # key clinical metric
    "T5_coherence":            (0.70, "med-pri"),
    "T5_pass_dual":            (1.0,  "low-pri"),
    "T6_velocity_cv":          None,               # descriptor only
    "T7_treatment_d":          (0.50, "HIGH-PRI"),
    "T8_kendall_tau":          (0.30, "HIGH-PRI"),  # narrowest CNN/ViT gap
}
LOWER_BETTER = {"H4_norm_cv"}

for mn in models:
    r = results.get(mn, {})
    passed = total = 0
    high_pri = []
    for k, tup in THRESHOLDS.items():
        if tup is None or k not in r: continue
        thresh, pri = tup
        total += 1
        ok = (r[k] <= thresh) if k in LOWER_BETTER else (r[k] >= thresh)
        if ok: passed += 1
        if pri == "HIGH-PRI" and not ok:
            high_pri.append((k, r[k], thresh))
    print("\n{}  [{}/{} pass]".format(mn, passed, total))
    for k, tup in THRESHOLDS.items():
        if tup is None:
            v = r.get(k)
            if v is not None: print("  {:<35s} {:>8.3f}  (descriptor/removed)".format(k, v))
            continue
        if k not in r: continue
        thresh, pri = tup
        ok = (r[k] <= thresh) if k in LOWER_BETTER else (r[k] >= thresh)
        flag = "PASS" if ok else "FAIL"
        print("  {:<35s} {:>8.3f}  thresh={:<6}  {}  [{}]".format(
              k, r[k], thresh, flag, pri))
    if high_pri:
        print("  HIGH-PRIORITY FAILS (Phase 3 targets):")
        for k, v, t in high_pri:
            print("    {} = {:.3f} -> needs > {:.2f}".format(k, v, t))

import json as _j
with open(OUTPUT_ROOT / "eval_results_final.json", "w") as f:
    _j.dump(results, f, indent=2)
print("\nSaved: eval_results_final.json")


In [ ]:
# STATIC CNN LIMITATIONS — quantitative evidence + priority summary
print("\n" + "="*60)
print("  VIT vs CNN COMPARISON — PHASE 3 CONCLUSION")
print("="*60)

for mn in models:
    r = results.get(mn, {})
    print("\nModel: {}".format(mn))

    print("  -- Morphology --")
    print("    M1 Vol R2 (Ridge):   {:.3f}  (linear fails; nonlinear below)".format(
          r.get("M1_volume_R2_ridge", 0)))
    print("    M1 Vol R2 (RF):      {:.3f}  (nonlinear probe; thresh > 0.50)".format(
          r.get("M1_volume_R2_rf", 0)))
    print("    M1 Spearman rho:     {:.3f}  (rank-order signal; thresh > 0.55)".format(
          r.get("M1_spearman_rho", 0)))
    print("    M4 Necrosis F1:      {:.3f}  (thresh > 0.60)".format(
          r.get("M4_necrosis_F1", 0)))
    print("    M6 Patient Purity:   {:.1f}%  (thresh > 60%) [HIGH-PRI]".format(
          r.get("M6_patient_purity_pct", 0)))

    print("  -- Heterogeneity --")
    print("    H1 RankMe:           {:.1f}  (thresh > 30)".format(
          r.get("H1_rankme", 0)))
    print("    H2 Diversity:        {:.3f}  (thresh > 0.25)".format(
          r.get("H2_diversity", 0)))
    print("    H3 Responder F1:     {:.3f}  (thresh > 0.55) -- SURPRISE".format(
          r.get("H3_responder_F1", 0)))
    print("    H4 Norm CV:          {:.3f}  (mitigated by L2-norm in eval)".format(
          r.get("H4_norm_cv", 0)))

    print("  -- Temporal (HIGH-PRIORITY for Phase 3) --")
    print("    T1 Spearman rho:     {:.3f}  (thresh > 0.30) [HIGH-PRI]".format(
          r.get("T1_spearman_wt", 0)))
    print("    T4 RANO AUC:         {:.3f}  (thresh > 0.65) [HIGH-PRI]".format(
          r.get("T4_rano_auc", 0)))
    print("    T7 Treatment d:      {:.3f}  (thresh > 0.50) [HIGH-PRI]".format(
          r.get("T7_treatment_d", 0)))
    print("    T8 Kendall tau:      {:.3f}  (thresh > 0.30) [HIGH-PRI]".format(
          r.get("T8_kendall_tau", 0)))

print()
print("PHASE 2 CONCLUSION")
print("-"*60)
print("CNN PlainConvUNet — Dice=0.832 — embeddings (1620, 1280)")
print()
print("STRENGTHS:")
print("  Representation richness:   RankMe=335 (target > 30) 11x")
print("  Necrosis detection:        M4 F1=0.722 (PASS)")
print("  Outcome prediction:        H3 F1=0.781 (PASS - surprise)")
print("  Volume rank-encoding:      M1 Spearman=0.637 (PASS)")
print("  Visit coherence:           T5=0.777 (PASS)")
print()
print("LIMITATIONS (Phase 3 targets):")
print("  M6 Patient purity:  8.2% -> target 70%  (ViT temporal training)")
print("  T4 RANO AUC:        0.521 -> target 0.65 (ViT progression detect)")
print("  T7 Cohen d:         0.201 -> target 0.50 (ViT treatment effect)")
print("  T8 Kendall tau:     0.251 -> target 0.30 (narrowest gap)")
print()
print("NOTE: T3 R2 removed from pass/fail — linear temporal prediction")
print("      inherently hard; use T1 Spearman + T8 tau instead.")
print()
print("=> Phase 3 SwinUNETR + temporal sequences targets these 4 metrics.")
print("Phase 2 COMPLETE")


# ── Final printed summary ──
print('\n' + '='*60)
print('  PHASE 3B SUMMARY — SwinUNETR ViT vs CNN')
print('='*60)
print('  CNN baseline: 16/26 tests pass (Phase 2)')
print('  SwinUNETR targets:')
print('    M6 patient purity:  CNN=7.3%   Target >70%')
print('    T4 RANO AUC:        CNN=0.526  Target >0.65')
print('    T7 Cohens d:        CNN=0.266  Target >0.50')
print('    T8 Kendall tau:     CNN=0.251  Target >0.30')
print('    T1 Spearman rho:    CNN=0.387  Target >0.40')
print('  -> Next: Phase 3C temporal sequences (T_max, 1929) for TD-ViT')
